# ASCAD fixed-key: exploratory analysis

Prereq: `python -m data.download_ascad` has produced `data/raw/ASCAD.h5`.

Covers: trace shape/amplitude, class balance of the 256-class S-box label,
and a per-sample **SNR** curve showing *where* in the 700-sample window the
(masked) first-round S-box leaks.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root on path

import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_ascad, summarise
from src.aes import AES_SBOX, hamming_weight
from src.preprocessing import build_labels, TraceScaler

data = load_ascad('../data/raw/ASCAD.h5', n_profiling=20000, n_attack=5000)
print(summarise(data))

In [ ]:
# --- a few raw traces + mean/std envelope ---
p = data.profiling
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for i in range(5):
    ax[0].plot(p.traces[i], lw=0.6, alpha=0.7)
ax[0].set_title('5 raw profiling traces'); ax[0].set_ylabel('amplitude (int8)')
m, s = p.traces.mean(0), p.traces.std(0)
ax[1].plot(m, label='mean'); ax[1].fill_between(range(p.n_samples), m-s, m+s, alpha=0.3, label='+/-1 std')
ax[1].set_title('per-sample mean / std across profiling set'); ax[1].set_xlabel('sample'); ax[1].legend()
plt.tight_layout()

In [ ]:
# --- class balance of the 256-class label (byte 2, ID model) ---
y = build_labels(p, target_byte=2, leakage_model='ID')
counts = np.bincount(y, minlength=256)
print(f'classes present: {(counts>0).sum()}/256   min/mean/max per class: {counts.min()}/{counts.mean():.0f}/{counts.max()}')
plt.figure(figsize=(11,3)); plt.bar(range(256), counts, width=1.0)
plt.title('label histogram (byte 2, ID) - should be roughly uniform'); plt.xlabel('Sbox(p^k) value');

In [ ]:
# --- where does it leak?  SNR of the HW(Sbox(p^k)) model per sample ---
# SNR(t) = Var_over_classes( E[trace_t | class] ) / Mean_over_classes( Var[trace_t | class] )
hw = hamming_weight(AES_SBOX[p.plaintext[:, 2] ^ p.key[:, 2]])
X = p.traces.astype(np.float64)
cls_means, cls_vars = [], []
for c in range(9):
    mask = hw == c
    if mask.sum() > 1:
        cls_means.append(X[mask].mean(0)); cls_vars.append(X[mask].var(0))
snr = np.var(np.array(cls_means), axis=0) / np.mean(np.array(cls_vars), axis=0)
plt.figure(figsize=(11,3)); plt.plot(snr)
plt.title('per-sample SNR, HW(Sbox(p^k)) model'); plt.xlabel('sample'); plt.ylabel('SNR')
print('argmax SNR sample:', int(np.argmax(snr)), ' max SNR:', float(snr.max()))

In [ ]:
# --- standardized traces: what the models actually see ---
sc = TraceScaler('standardize').fit(p.traces)
xs = sc.transform(p.traces[:2000])
print('after standardize: mean ~', xs.mean().round(4), ' std ~', xs.std().round(4))
plt.figure(figsize=(11,3)); plt.plot(xs[:3].T, lw=0.6); plt.title('standardized traces');

### Takeaways to expect
* Labels are ~uniform over the 256 classes (no class-imbalance shortcut).
* The ID-model SNR is very low everywhere (masking); the HW model shows a
  small bump where the S-box output is manipulated - that is the leakage the
  CNN has to find. This is why raw accuracy is near chance yet the key rank
  still converges.